In [0]:
providers = spark.table("bronze_providers")
members = spark.table("bronze_members")
claims = spark.table("bronze_claims")

print("Loaded Bronze tables")

In [0]:
silver_claims = spark.table("silver_claims")

new_claims = claims.join(
    silver_claims.select("claim_id"),
    on="claim_id",
    how="left_anti"
)

print("New claims to process:", new_claims.count())

In [0]:
print("New claims to process:", new_claims.count())
display(new_claims.orderBy("claim_id"))

In [0]:
display(new_claims.select(
    "claim_id", "member_id", "provider_id", "claim_amount"
))

In [0]:
from pyspark.sql import functions as F
new_claims = new_claims.withColumn("claim_date", F.to_date("claim_date")) \
               .withColumn("claim_amount", F.col("claim_amount").cast("double"))

members = members.withColumn("date_of_birth", F.to_date("date_of_birth"))

new_claims.printSchema()
members.printSchema()

In [0]:
providers = providers.withColumn("region", F.upper("region"))
members = members.withColumn("region", F.upper("region"))

In [0]:
new_claims = new_claims.dropDuplicates(["claim_id"])

print("Claims after deduplication:", new_claims.count())

In [0]:
invalid_amounts=  new_claims.filter(F.col("claim_amount") < 0)

print("invalid amount:",display( invalid_amounts))

In [0]:
invalid_members = new_claims.join(members.select("member_id"), on="member_id", how="left_anti")

display(invalid_members)


In [0]:
invalid_providers = new_claims.join(providers.select("provider_id"), on="provider_id", how="left_anti")

display(invalid_providers)

In [0]:
valid_claims = new_claims.filter((F.col("claim_amount") > 0) & F.col("member_id").isNotNull() & F.col("provider_id").isNotNull()).join (members.select("member_id"), on="member_id", how="inner").join(providers.select("provider_id"), on="provider_id", how="inner")
print("valid_claims:",valid_claims.count())

In [0]:
providers.write.mode("overwrite").format("delta").saveAsTable("silver_provider")
members.write.mode("overwrite").format("delta").saveAsTable("silver_member")
valid_claims.write.mode("append").format("delta").saveAsTable("silver_claims")

print("silver tables created successfully")

In [0]:
print("Providers:", spark.table("silver_provider").count())
print("Members:", spark.table("silver_member").count())
print("Claims:", spark.table("silver_claims").count())